# Tutor Inteligente — OCR Espanol + Detector YOLO
**Kaggle Dual T4 GPU | Marzo 2026**

Pipeline completo:
1. Construir `char_map.json` desde `yolo_dataset_final`
2. Extraer crops del dataset YOLO para el Clasificador (EfficientNet-B2)
3. Entrenar Clasificador con DataParallel 2xT4, AMP FP16, MLflow
4. Exportar Clasificador a ONNX opset 17 + INT8
5. Entrenar Detector YOLOv8n con MLflow (mismo tracking URI)
6. Exportar Detector a ONNX 640x640
7. Copiar artefactos + paquete ZIP final

**Dataset:** `yolo_dataset_final` subido a Kaggle
**MLflow:** ambos entrenamientos en el mismo `./mlruns`


In [ ]:
!pip install timm
!pip install onnx
!pip install onnxruntime-gpu
!pip install onnxruntime-tools
!pip install onnxsim
!pip install albumentations
!pip install tqdm
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn
!pip install onnxconverter-common
!pip install mlflow
!pip install ultralytics
!pip install pillow-avif
!pip install yalm

In [ ]:
import torch
import timm
import onnx
import onnxruntime
import mlflow
import ultralytics

print(f'PyTorch    : {torch.__version__}')
print(f'TIMM       : {timm.__version__}')
print(f'ONNX       : {onnx.__version__}')
print(f'ORT        : {onnxruntime.__version__}')
print(f'MLflow     : {mlflow.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'CUDA       : {torch.version.cuda}')

In [ ]:
import os, random, time, json, shutil, re
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter

# 1. Configuración de Semillas y Dispositivo
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'🚀 Dispositivo: {DEVICE} | GPUs detectadas: {NUM_GPUS}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'   - GPU {i}: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB')

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# 2. LOCALIZACIÓN DIRECTA DEL DATASET (Sin rglob lento)
# Usamos la ruta exacta que proporcionaste
ruta_directa = Path('/kaggle/input/datasets/danielperegrinoperez/yolo-ocr-espaol/yolo_dataset_final/dataset.yaml')

if ruta_directa.exists():
    _DATASET_ROOT = ruta_directa
    print(f"✅ Dataset encontrado en ruta directa: {_DATASET_ROOT}")
else:
    print("🔍 Ruta directa no encontrada, buscando en subdirectorios de primer nivel...")
    # Búsqueda limitada (solo un nivel de profundidad para ser rápido)
    _DATASET_ROOT = None
    for p in Path('/kaggle/input').iterdir():
        cand = p / 'yolo_dataset_final'
        if cand.exists():
            _DATASET_ROOT = cand
            break
    
    if _DATASET_ROOT is None:
        # Último intento en working
        _DATASET_ROOT = Path('/kaggle/working/yolo_dataset_final')

# Verificación de Seguridad
if not _DATASET_ROOT.exists():
    print("\n❌ ERROR: No se encontró el directorio del dataset.")
    print("Contenido de /kaggle/input para depurar:", os.listdir('/kaggle/input'))
    raise AssertionError("Dataset no encontrado.")

# 3. Definición de Rutas
DATASET_YAML = str(_DATASET_ROOT / 'dataset.yaml')
IMAGES_TRAIN = _DATASET_ROOT / 'images' / 'train'
IMAGES_VAL   = _DATASET_ROOT / 'images' / 'val'
LABELS_TRAIN = _DATASET_ROOT / 'labels' / 'train'
LABELS_VAL   = _DATASET_ROOT / 'labels' / 'val'

print(f"📍 Ruta del YAML: {DATASET_YAML}")

# 4. Preparación de Directorios de Salida
OUTPUT_DIR = Path('/kaggle/working')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
ONNX_DIR   = OUTPUT_DIR / 'onnx'
YOLO_DIR   = OUTPUT_DIR / 'runs' / 'detect'
MLRUNS_DIR = str(OUTPUT_DIR / 'mlruns')

for d in [CKPT_DIR, ONNX_DIR, YOLO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 5. Configuración de Modelos
CFG = dict(
    img_size=64, num_workers=4, pin_memory=True,
    model_name='efficientnet_b2', pretrained=True, drop_rate=0.3,
    epochs=60, batch_size=256, lr=3e-3, weight_decay=1e-4,
    ckpt_dir=str(CKPT_DIR), onnx_dir=str(ONNX_DIR),
    mlflow_uri=MLRUNS_DIR, mlflow_experiment='tutor_inteligente_ocr',
)

YOLO_CFG = dict(
    model_variant='yolov8n.pt', epochs=80, batch=32,
    device='0,1' if NUM_GPUS >= 2 else '0',
    workers=8, img_size=640, lr0=0.02, lrf=0.01,
    project=str(YOLO_DIR), name='char_detector_t4',
    cache='ram'
)

print(f'\n✅ Configuración lista.')
print(f'   - Batch Clasificador: {CFG["batch_size"] * max(NUM_GPUS,1)}')
print(f'   - Dispositivo YOLO: {YOLO_CFG["device"]}')

## Celda 3 — Construir char_map.json desde filenames del dataset

In [ ]:
# =============================================================================
# CELDA 3 - Construir char_map.json con las 107 clases reales
#
# FUENTE PRIMARIA: dataset_classes_report.json subido a Kaggle.
# Contiene la lista canonica de todas las clases objetivo (107).
#
# RAIZ DEL PROBLEMA con la version anterior:
# prepare_yolo_dataset.py renombra archivos como {original_stem}_{i:06d}.
# Los archivos de Spanish/HWC cuya clase estaba en el NOMBRE DE LA CARPETA
# (ej: carpeta 'n/' con 'Sample001.jpg') quedan como 'Sample001_000042.jpg'.
# El filename no tiene la clase -> scan de filenames solo detecta 62 (EMNIST).
# La solucion: leer el reporte JSON como fuente de verdad de las 107 clases.
# =============================================================================

EMNIST_CHARS = list('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
PRIM_SLUG_MAP = {
    'linea_vertical'         : 'linea_vertical',
    'linea_horizontal'       : 'linea_horizontal',
    'linea_oblicua_derecha'  : 'linea_oblicua_derecha',
    'linea_oblicua_izquierda': 'linea_oblicua_izquierda',
    'curva'                  : 'curva',
    'circulo'                : 'circulo',
}

# ---- 1. Localizar dataset_classes_report.json en /kaggle/input/ ----
_REPORT_PATH = None
for candidate in Path('/kaggle/input').rglob('dataset_classes_report.json'):
    _REPORT_PATH = candidate
    break
if _REPORT_PATH is None:
    _fallback = Path('/kaggle/working/dataset_classes_report.json')
    if _fallback.exists():
        _REPORT_PATH = _fallback
print(f'dataset_classes_report.json : {_REPORT_PATH}')

# ---- 2. Cargar clases objetivo desde el reporte ----
if _REPORT_PATH and _REPORT_PATH.exists():
    with open(_REPORT_PATH, 'r', encoding='utf-8') as f:
        _report = json.load(f)
    target_classes  = _report.get('target_classes', [])
    _global_missing = set(_report.get('global_missing', []))
    _coverage       = _report.get('coverage_summary', {})
    print(f'Reporte: {len(target_classes)} clases objetivo | faltantes={len(_global_missing)} | cobertura={_coverage.get("coverage_pct","?")}')
else:
    print('[WARN] dataset_classes_report.json no encontrado. Usando EMNIST+primitivos como fallback.')
    target_classes  = EMNIST_CHARS + list(PRIM_SLUG_MAP.values())
    _global_missing = set(PRIM_SLUG_MAP.values())

# ---- 3. Contar imagenes decodificables por clase desde filenames ----
# Solo los archivos sinteticos cls*/prim*/char_* tienen la clase en el nombre.
# Los archivos de Spanish/HWC reales NO son decodificables (ver nota arriba).
def decode_char_from_stem(stem):
    m = re.match(r'^cls(\d{3})_', stem)
    if m:
        idx = int(m.group(1))
        return (EMNIST_CHARS[idx], True) if idx < len(EMNIST_CHARS) else (None, False)
    m = re.match(r'^prim_([a-z_]+)_', stem)
    if m and m.group(1) in PRIM_SLUG_MAP:
        return PRIM_SLUG_MAP[m.group(1)], True
    m = re.match(r'^char_(\d{2})_', stem)
    if m:
        idx = int(m.group(1))
        return (EMNIST_CHARS[idx], True) if idx < len(EMNIST_CHARS) else (None, False)
    m = re.match(r'^(.)_\d', stem)
    if m and m.group(1).strip() and len(m.group(1)) == 1:
        return m.group(1), False
    return None, False

print('Escaneando filenames en images/train ...')
char_freq_from_files = Counter()
all_stems = [p.stem for p in sorted(IMAGES_TRAIN.glob('*.jpg'))]
for stem in all_stems:
    char, _ = decode_char_from_stem(stem)
    if char:
        char_freq_from_files[char] += 1

total_imgs  = len(all_stems)
n_decodable = sum(char_freq_from_files.values())
print(f'  Total imagenes train     : {total_imgs:,}')
print(f'  Con clase decodificable  : {n_decodable:,}  ({n_decodable*100//max(total_imgs,1)}%)')
print(f'  Sin clase decodificable  : {total_imgs - n_decodable:,}  (Spanish/HWC renombrados)')
print(f'  Clases en filenames      : {len(char_freq_from_files)}')

# ---- 4. Construir char_map unificado con las 107 clases del reporte ----
seen = set(); ordered_chars = []
for c in target_classes:
    if c not in seen:
        seen.add(c); ordered_chars.append(c)

CHAR2IDX    = {c: i for i, c in enumerate(ordered_chars)}
IDX2CHAR    = {i: c for i, c in enumerate(ordered_chars)}
NUM_CLASSES = len(ordered_chars)

n_emnist = sum(1 for c in ordered_chars if c in set(EMNIST_CHARS))
n_prim   = sum(1 for c in ordered_chars if c in set(PRIM_SLUG_MAP.values()))
n_extra  = NUM_CLASSES - n_emnist - n_prim

classes_with_data    = [c for c in ordered_chars if char_freq_from_files.get(c, 0) > 0]
classes_without_data = [c for c in ordered_chars if char_freq_from_files.get(c, 0) == 0]

print(f'\nchar_map desde reporte:')
print(f'  NUM_CLASSES   : {NUM_CLASSES}')
print(f'  EMNIST        : {n_emnist}')
print(f'  Primitivos    : {n_prim}')
print(f'  Extras espanol: {n_extra}')
print(f'  Primeras 20   : {ordered_chars[:20]}')
print(f'  Ultimas  10   : {ordered_chars[-10:]}')
print(f'  Con crops decodificables : {len(classes_with_data)}/{NUM_CLASSES}')
print(f'  Sin crops decodificables : {len(classes_without_data)} -> no entrenaran el clasificador')
if classes_without_data:
    print(f'    Clases sin datos: {classes_without_data[:30]}')
    print('    FIX: re-ejecutar generate_synthetic_yolo.py para esas clases')
    print('    y preparar el dataset incluyendo esos sinteticos nuevos.')

# ---- 5. Guardar char_map.json ----
char_map_path = ONNX_DIR / 'char_map.json'
with open(char_map_path, 'w', encoding='utf-8') as f:
    json.dump({
        'idx2char'         : {str(i): c for i, c in IDX2CHAR.items()},
        'char2idx'         : CHAR2IDX,
        'num_classes'      : NUM_CLASSES,
        'class_freq'       : {c: char_freq_from_files.get(c, 0) for c in ordered_chars},
        'trainable_classes': len(classes_with_data),
        'source'           : str(_REPORT_PATH) if _REPORT_PATH else 'fallback',
    }, f, ensure_ascii=False, indent=2)
print(f'\nchar_map.json guardado -> {char_map_path}  ({NUM_CLASSES} clases)')


## Celda 4 — Dataset de crops para el clasificador

In [ ]:
import torchvision
from pathlib import Path

# 1. Definir EMNIST_ROOT en una zona con permisos de escritura
EMNIST_ROOT = Path('/kaggle/working/data/emnist')
EMNIST_ROOT.mkdir(parents=True, exist_ok=True)

print("=== Descargando EMNIST (ByClass) ===")
# Esto descargará el dataset si no existe
try:
    train_ds = torchvision.datasets.EMNIST(
        root=str(EMNIST_ROOT), 
        split='byclass', 
        train=True, 
        download=True
    )
    print(f"✅ EMNIST descargado con éxito en: {EMNIST_ROOT}")
except Exception as e:
    print(f"❌ Error al descargar EMNIST: {e}")

# 2. Asegurar que las rutas de tu dataset YOLO existen
# Usamos las variables definidas en la celda de detección de GPU
if '_DATASET_ROOT' in locals():
    IMAGES_TRAIN = _DATASET_ROOT / 'images' / 'train'
    LABELS_TRAIN = _DATASET_ROOT / 'labels' / 'train'
    IMAGES_VAL   = _DATASET_ROOT / 'images' / 'val'
    LABELS_VAL   = _DATASET_ROOT / 'labels' / 'val'
    print(f"✅ Rutas de imágenes configuradas: {IMAGES_TRAIN}")
else:
    print("❌ ERROR: No se encontró _DATASET_ROOT. Ejecuta la celda de las GPUs primero.")

In [ ]:
"""
CELDA CORREGIDA — Fuente de datos del clasificador
===================================================
Reemplaza las celdas 3-B / 4-B del notebook.

Problemas que resuelve:
  1. IMAGES_TRAIN con 0 archivos → debug de ruta + rglob
  2. 45 clases sin datos (ñ, tildes, símbolos, primitivos) →
     generación sintética con Pillow + fuentes del sistema
  3. FILE2CHAR no definido → eliminado
  4. decode_char_robust no definido → incluido aquí
"""

# ── Imports ───────────────────────────────────────────────────────────────
import re, random, io, cv2, numpy as np, torchvision
from pathlib import Path
from collections import Counter
from PIL import Image as PilImage, ImageDraw, ImageFont

random.seed(42); np.random.seed(42)

# ── Constantes de decodificación ──────────────────────────────────────────
EMNIST_CHARS = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
PRIM_SLUG_MAP = {
    "linea_vertical"         : "línea_vertical",
    "linea_horizontal"       : "línea_horizontal",
    "linea_oblicua_derecha"  : "línea_oblicua_derecha",
    "linea_oblicua_izquierda": "línea_oblicua_izquierda",
    "curva"                  : "curva",
    "circulo"                : "círculo",
}

# =============================================================================
# PASO 0 — Debug: confirmar que IMAGES_TRAIN apunta al lugar correcto
# =============================================================================
print("=" * 60)
print("  DIAGNÓSTICO DE RUTAS")
print("=" * 60)

# Buscar dataset.yaml en todas las rutas posibles de Kaggle
_yaml_candidates = list(Path("/kaggle/input").rglob("dataset.yaml"))
_yaml_candidates += list(Path("/kaggle/working").rglob("dataset.yaml"))
print(f"\ndataset.yaml encontrados: {len(_yaml_candidates)}")
for p in _yaml_candidates:
    print(f"  {p}")

# Usar la primera si _DATASET_ROOT no está bien apuntada
if not hasattr(Path, '_checked'):
    for candidate_yaml in _yaml_candidates:
        candidate_root = candidate_yaml.parent
        candidate_train = candidate_root / "images" / "train"
        n = sum(1 for _ in candidate_train.rglob("*.jpg")) if candidate_train.exists() else 0
        print(f"\n  Candidato: {candidate_root}")
        print(f"    images/train existe : {candidate_train.exists()}")
        print(f"    *.jpg en train      : {n}")
        if n > 0:
            # Sobrescribir las variables del notebook si apuntaban mal
            _DATASET_ROOT = candidate_root
            IMAGES_TRAIN  = candidate_train
            IMAGES_VAL    = candidate_root / "images" / "val"
            LABELS_TRAIN  = candidate_root / "labels" / "train"
            LABELS_VAL    = candidate_root / "labels" / "val"
            DATASET_YAML  = str(candidate_yaml)
            print(f"    ✅ USANDO ESTE DATASET ROOT")
            break

# Verificación final de rutas
print(f"\n  IMAGES_TRAIN : {IMAGES_TRAIN}")
print(f"  Existe       : {IMAGES_TRAIN.exists()}")
n_train_total = sum(1 for _ in IMAGES_TRAIN.rglob("*.jpg")) if IMAGES_TRAIN.exists() else 0
print(f"  .jpg (rglob) : {n_train_total:,}")

if n_train_total == 0:
    print("\n  ⚠  Sin imágenes en train — se usará solo EMNIST + síntesis Pillow")

# Mostrar muestra de nombres para entender el patrón
if IMAGES_TRAIN.exists():
    muestra = list(IMAGES_TRAIN.rglob("*.jpg"))[:8]
    print(f"\n  Muestra de nombres ({min(8,len(muestra))}):")
    for f in muestra:
        print(f"    {f.name}")


# =============================================================================
# PASO 1 — Funciones auxiliares
# =============================================================================

def decode_char_from_stem(stem: str) -> str | None:
    """Decodifica la clase desde el stem del archivo si el nombre la codifica."""
    # cls{idx:03d}_*  (generate_synthetic_yolo.py)
    m = re.match(r'^cls(\d{3})_', stem)
    if m:
        idx = int(m.group(1))
        return EMNIST_CHARS[idx] if idx < len(EMNIST_CHARS) else None
    # prim_{slug}_*
    m = re.match(r'^prim_([a-z_]+)_', stem)
    if m and m.group(1) in PRIM_SLUG_MAP:
        return PRIM_SLUG_MAP[m.group(1)]
    # char_{idx:02d}_*  (compatibilidad)
    m = re.match(r'^char_(\d{2})_', stem)
    if m:
        idx = int(m.group(1))
        return EMNIST_CHARS[idx] if idx < len(EMNIST_CHARS) else None
    return None


def extract_crops_from_image(img_path: Path, img_size: int) -> list[np.ndarray]:
    """
    Extrae crops RGB de una imagen usando su label YOLO si existe.
    Devuelve imagen completa si no hay label.
    """
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return []

    # Buscar label en labels/train/ o junto a la imagen
    lbl_candidates = [
        LABELS_TRAIN / (img_path.stem + ".txt"),
        img_path.with_suffix(".txt"),
    ]
    lbl_path = next((p for p in lbl_candidates if p.exists()), None)

    if lbl_path and lbl_path.exists():
        lines = lbl_path.read_text().strip().splitlines()
        H, W  = img_bgr.shape[:2]
        crops = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5: continue
            xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = max(0, int((xc - bw/2)*W));  y1 = max(0, int((yc - bh/2)*H))
            x2 = min(W, int((xc + bw/2)*W));  y2 = min(H, int((yc + bh/2)*H))
            if x2 <= x1 or y2 <= y1: continue
            crop = cv2.cvtColor(img_bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
            crops.append(cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_CUBIC))
        if crops:
            return crops

    # Sin label: imagen completa
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return [cv2.resize(rgb, (img_size, img_size), interpolation=cv2.INTER_CUBIC)]


def render_char_with_pillow(
    char: str,
    size: int = 64,
    n_variants: int = 1,
) -> list[np.ndarray]:
    """
    Genera imágenes sintéticas de un carácter usando Pillow + fuentes del sistema.
    Produce fondo BLANCO, tinta NEGRA (igual distribución que EMNIST invertido).

    Útil para: ñ, Ñ, á, é, í, ó, ú, Á, É, Í, Ó, Ú, ü, Ü, símbolos, primitivos.
    """
    # Fuentes candidatas (Windows + Linux Kaggle)
    font_candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
        "/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf",
        "/usr/share/fonts/truetype/ubuntu/Ubuntu-R.ttf",
        "C:/Windows/Fonts/arial.ttf",
        "C:/Windows/Fonts/calibri.ttf",
    ]

    font = None
    font_size = int(size * 0.68)
    for fp in font_candidates:
        if Path(fp).exists():
            try:
                font = ImageFont.truetype(fp, font_size)
                break
            except Exception:
                continue

    if font is None:
        # Último recurso: fuente bitmap de Pillow (solo ASCII básico)
        try:
            font = ImageFont.load_default(size=font_size)
        except Exception:
            font = ImageFont.load_default()

    results = []

    # Primitivos → dibujar formas con OpenCV en lugar de texto
    if char in ("línea_vertical", "línea_horizontal", "línea_oblicua_derecha",
                "línea_oblicua_izquierda", "curva", "círculo"):
        for _ in range(n_variants):
            canvas = np.full((size, size, 3), 255, dtype=np.uint8)
            m      = int(size * 0.15)
            thick  = max(1, size // 16) + random.randint(0, 2)
            if char == "línea_vertical":
                x = random.randint(m, size-m)
                cv2.line(canvas, (x, m), (x, size-m), (0,0,0), thick)
            elif char == "línea_horizontal":
                y = random.randint(m, size-m)
                cv2.line(canvas, (m, y), (size-m, y), (0,0,0), thick)
            elif char == "línea_oblicua_derecha":
                cv2.line(canvas, (m, size-m), (size-m, m), (0,0,0), thick)
            elif char == "línea_oblicua_izquierda":
                cv2.line(canvas, (m, m), (size-m, size-m), (0,0,0), thick)
            elif char == "curva":
                ax = random.randint(size//5, size//3)
                ay = random.randint(size//6, size//4)
                a0 = random.randint(0, 90)
                cv2.ellipse(canvas, (size//2, size//2), (ax, ay),
                            random.randint(0,360), a0, a0+random.randint(120,270), (0,0,0), thick)
            elif char == "círculo":
                r = random.randint(size//5, size//3)
                cv2.circle(canvas, (size//2, size//2), r, (0,0,0), thick)
            # Blur leve para naturalidad
            canvas = cv2.GaussianBlur(canvas, (3,3), 0.5)
            results.append(canvas)
        return results

    # Caracteres de texto → Pillow
    for i in range(n_variants):
        # Variar tamaño de fuente levemente
        f_size = font_size + random.randint(-4, 4)
        try:
            f = ImageFont.truetype(font.path, f_size) if hasattr(font, 'path') else font
        except Exception:
            f = font

        img = PilImage.new("RGB", (size, size), (255, 255, 255))
        draw = ImageDraw.Draw(img)

        # Calcular posición centrada
        bbox = draw.textbbox((0, 0), char, font=f)
        x = (size - (bbox[2] - bbox[0])) // 2 - bbox[0]
        y = (size - (bbox[3] - bbox[1])) // 2 - bbox[1]

        # Variar posición y grosor de trazo levemente
        x += random.randint(-3, 3)
        y += random.randint(-3, 3)

        draw.text((x, y), char, fill=(random.randint(0,40), 0, 0), font=f)

        arr = np.array(img)
        # Blur leve
        arr = cv2.GaussianBlur(arr, (3,3), 0.4)
        results.append(arr)

    return results


# =============================================================================
# PASO 2 — Fuente 1: Dataset YOLO (sintéticos cls{idx}_* y prim_*)
# =============================================================================
print("\n" + "=" * 60)
print("  FUENTE 1: Imágenes sintéticas en IMAGES_TRAIN")
print("=" * 60)

synthetic_samples: list[tuple[np.ndarray, int]] = []
skipped_no_class = 0

# rglob para cubrir subdirectorios (por si acaso)
all_train_imgs = list(IMAGES_TRAIN.rglob("*.jpg"))
print(f"\n  Archivos encontrados (rglob): {len(all_train_imgs):,}")

for img_path in all_train_imgs:
    char = decode_char_from_stem(img_path.stem)
    if char is None:
        skipped_no_class += 1
        continue
    if char not in CHAR2IDX:
        continue
    crops = extract_crops_from_image(img_path, CFG['img_size'])
    for crop in crops:
        synthetic_samples.append((crop, CHAR2IDX[char]))

syn_counts = Counter(s[1] for s in synthetic_samples)
print(f"  Con clase deducible: {len(all_train_imgs) - skipped_no_class:,}")
print(f"  Sin clase (reales) : {skipped_no_class:,}")
print(f"  Muestras extraídas : {len(synthetic_samples):,}")
print(f"  Clases cubiertas   : {len(syn_counts)}/{NUM_CLASSES}")


# =============================================================================
# PASO 3 — Fuente 2: EMNIST torchvision (62 clases limpias)
# =============================================================================
print("\n" + "=" * 60)
print("  FUENTE 2: EMNIST torchvision")
print("=" * 60)

emnist_samples: list[tuple[np.ndarray, int]] = []

# Calcular cap: equilibrar con sintéticos
syn_max    = max(syn_counts.values()) if syn_counts else 0
EMNIST_CAP = max(syn_max, 300)   # al menos 300 por clase
print(f"\n  Cap por clase: {EMNIST_CAP}")

try:
    emnist_ds = torchvision.datasets.EMNIST(
        root=str(EMNIST_ROOT), split="byclass", train=True, download=False
    )
    emnist_idx2char = (
        [str(d) for d in range(10)] +
        [chr(c) for c in range(ord("A"), ord("Z") + 1)] +
        [chr(c) for c in range(ord("a"), ord("z") + 1)]
    )
    # EMNIST idx → char_map idx
    emnist_to_clf = {
        ei: CHAR2IDX[ch]
        for ei, ch in enumerate(emnist_idx2char)
        if ch in CHAR2IDX
    }
    # Agrupar por clase del char_map
    emnist_by_clf: dict[int, list[int]] = {}
    for i, lbl in enumerate(emnist_ds.targets):
        ci = emnist_to_clf.get(int(lbl))
        if ci is not None:
            emnist_by_clf.setdefault(ci, []).append(i)

    for ci, indices in emnist_by_clf.items():
        # Si ya hay suficientes sintéticos para esta clase, usar menos EMNIST
        already = syn_counts.get(ci, 0)
        need    = max(0, EMNIST_CAP - already)
        chosen  = random.sample(indices, min(len(indices), need))
        for emnist_i in chosen:
            img_pil, _ = emnist_ds[emnist_i]
            arr = np.array(img_pil)
            arr = cv2.flip(cv2.transpose(arr), flipCode=1)   # orientación
            arr = cv2.bitwise_not(arr)                        # fondo claro, tinta oscura
            rgb = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
            rsz = cv2.resize(rgb, (CFG['img_size'], CFG['img_size']), interpolation=cv2.INTER_CUBIC)
            emnist_samples.append((rsz, ci))

    emn_counts = Counter(s[1] for s in emnist_samples)
    print(f"  Muestras EMNIST  : {len(emnist_samples):,}")
    print(f"  Clases cubiertas : {len(emn_counts)}/{NUM_CLASSES}")

except Exception as e:
    print(f"  [WARN] EMNIST no disponible: {e}")
    emnist_samples = []


# =============================================================================
# PASO 4 — Fuente 3: Pillow synthesis para las 45 clases faltantes
# =============================================================================
print("\n" + "=" * 60)
print("  FUENTE 3: Síntesis Pillow (clases sin datos reales)")
print("=" * 60)

# Clases que siguen sin datos después de sintéticos + EMNIST
combined_so_far = Counter(s[1] for s in synthetic_samples + emnist_samples)
missing_classes = [IDX2CHAR[i] for i in range(NUM_CLASSES) if i not in combined_so_far]

PILLOW_N = EMNIST_CAP   # misma cantidad que EMNIST para no desbalancear

print(f"\n  Clases sin datos  : {len(missing_classes)}")
if missing_classes:
    print(f"  → {missing_classes[:20]}" + (" ..." if len(missing_classes) > 20 else ""))
print(f"  Generando {PILLOW_N} variantes por clase con Pillow...")

pillow_samples: list[tuple[np.ndarray, int]] = []
failed_chars: list[str] = []

for char in missing_classes:
    if char not in CHAR2IDX:
        continue
    ci      = CHAR2IDX[char]
    renders = render_char_with_pillow(char, size=CFG['img_size'], n_variants=PILLOW_N)
    if not renders:
        failed_chars.append(char)
        continue
    for img in renders:
        pillow_samples.append((img, ci))

pil_counts = Counter(s[1] for s in pillow_samples)
print(f"\n  Muestras Pillow   : {len(pillow_samples):,}")
print(f"  Clases generadas  : {len(pil_counts)}")
if failed_chars:
    print(f"  No generadas      : {failed_chars}")


# =============================================================================
# PASO 5 — Combinar todas las fuentes
# =============================================================================
print("\n" + "=" * 60)
print("  RESUMEN FINAL")
print("=" * 60)

train_samples = synthetic_samples + emnist_samples + pillow_samples

all_counts = Counter(s[1] for s in train_samples)
print(f"\n  Sintéticos  : {len(synthetic_samples):,}")
print(f"  EMNIST      : {len(emnist_samples):,}")
print(f"  Pillow      : {len(pillow_samples):,}")
print(f"  ─────────────────────")
print(f"  TOTAL       : {len(train_samples):,}")
print(f"  Clases      : {len(all_counts)}/{NUM_CLASSES}")

if all_counts:
    vals = sorted(all_counts.values())
    print(f"  Min/Med/Max : {vals[0]} / {vals[len(vals)//2]} / {vals[-1]}")
    sin_datos = [IDX2CHAR[i] for i in range(NUM_CLASSES) if i not in all_counts]
    if sin_datos:
        print(f"  Sin datos   : {sin_datos}")
    else:
        print(f"  ✅ TODAS las {NUM_CLASSES} clases tienen datos")

if len(train_samples) == 0:
    raise RuntimeError("❌ train_samples vacío. Revisa las rutas de IMAGES_TRAIN y EMNIST_ROOT.")


# =============================================================================
# PASO 6 — Val split
# =============================================================================
val_samples: list[tuple[np.ndarray, int]] = []

# Intentar val del dataset YOLO primero
for img_path in IMAGES_VAL.rglob("*.jpg"):
    char = decode_char_from_stem(img_path.stem)
    if char is None or char not in CHAR2IDX: continue
    crops = extract_crops_from_image(img_path, CFG['img_size'])
    for crop in crops:
        val_samples.append((crop, CHAR2IDX[char]))

if len(val_samples) == 0:
    # Fallback: split estratificado de train
    from sklearn.model_selection import StratifiedShuffleSplit
    _lbl = np.array([s[1] for s in train_samples])
    # Solo estratificar si todas las clases tienen ≥2 muestras
    _counts = Counter(_lbl)
    if min(_counts.values()) >= 2:
        _sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
        _tr, _va = next(_sss.split(train_samples, _lbl))
    else:
        # Split simple si alguna clase tiene 1 muestra
        _idx = list(range(len(train_samples)))
        random.shuffle(_idx)
        _cut = int(len(_idx) * 0.10)
        _va, _tr = _idx[:_cut], _idx[_cut:]
    val_samples   = [train_samples[i] for i in _va]
    train_samples = [train_samples[i] for i in _tr]
    print(f"\n  Val desde split   : {len(val_samples):,}")

print(f"\n  Train final : {len(train_samples):,}")
print(f"  Val final   : {len(val_samples):,}")
print("=" * 60)

In [ ]:
import cv2, random, numpy as np
from pathlib import Path
from collections import Counter
 
random.seed(42); np.random.seed(42)
 
# =============================================================================
# FUENTE EXTRA: Crops reales de images/train/ usando sus labels YOLO
# =============================================================================
# Estas imágenes son 640×640 con letra sobre fondo de libreta.
# Extraer el crop del bbox → mismo pipeline que en producción (processor.py).
# Esto enseña al modelo cómo lucen las letras en CONTEXTO REAL.
# =============================================================================
 
print("=== FUENTE EXTRA: Crops reales de imágenes YOLO ===\n")
 
REAL_CROPS_PER_CLASS = 80   # no demasiados para no dominar sobre EMNIST
real_crop_samples: list[tuple[np.ndarray, int]] = []
real_count_by_class: dict[int, int] = {}
 
all_imgs = list(IMAGES_TRAIN.rglob("*.jpg"))
print(f"  Escaneando {len(all_imgs):,} imágenes YOLO...")
 
for img_path in all_imgs:
    # Solo procesar imágenes cuya clase se puede deducir del nombre
    stem = img_path.stem
    char = decode_char_from_stem(stem)   # función definida en cell_data_sources_fixed.py
    if char is None or char not in CHAR2IDX:
        continue
 
    ci = CHAR2IDX[char]
    if real_count_by_class.get(ci, 0) >= REAL_CROPS_PER_CLASS:
        continue   # ya tenemos suficientes de esta clase
 
    lbl_path = LABELS_TRAIN / (stem + ".txt")
    if not lbl_path.exists():
        continue
 
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
 
    lines = lbl_path.read_text().strip().splitlines()
    H, W  = img_bgr.shape[:2]
 
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5: continue
        xc,yc,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
        x1=max(0,int((xc-bw/2)*W)); y1=max(0,int((yc-bh/2)*H))
        x2=min(W,int((xc+bw/2)*W)); y2=min(H,int((yc+bh/2)*H))
        if x2<=x1 or y2<=y1: continue
 
        crop_bgr = img_bgr[y1:y2, x1:x2]
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        crop_rsz = cv2.resize(crop_rgb, (CFG['img_size'], CFG['img_size']),
                              interpolation=cv2.INTER_CUBIC)
        real_crop_samples.append((crop_rsz, ci))
        real_count_by_class[ci] = real_count_by_class.get(ci, 0) + 1
        break   # 1 crop por imagen (variedad)
 
rc_counts = Counter(s[1] for s in real_crop_samples)
print(f"  Crops reales extraídos : {len(real_crop_samples):,}")
print(f"  Clases cubiertas       : {len(rc_counts)}")
 
# Mezclar en train_samples (ya definido por cell_data_sources_fixed.py)
train_samples = train_samples + real_crop_samples
print(f"  train_samples total    : {len(train_samples):,}")
print(f"  ✅ Crops reales añadidos — gap de distribución reducido")

In [ ]:
import re, random, cv2, numpy as np, torch, torch.nn as nn, time
from collections import Counter
from pathlib import Path
import torchvision
 
random.seed(42); np.random.seed(42); torch.manual_seed(42)
 
print("=== Construyendo val set LIMPIO (solo EMNIST) ===")
 
EMNIST_IDX2CHAR = (
    [str(d) for d in range(10)] +
    [chr(c) for c in range(ord("A"), ord("Z") + 1)] +
    [chr(c) for c in range(ord("a"), ord("z") + 1)]
)
EMNIST_VAL_PER_CLASS = 50   # muestras de val por clase EMNIST (3100 total)
 
val_emnist_samples: list[tuple[np.ndarray, int]] = []
 
try:
    _emnist_val_ds = torchvision.datasets.EMNIST(
        root=str(EMNIST_ROOT), split="byclass", train=False, download=False
    )
    _emnist_to_clf = {
        ei: CHAR2IDX[ch]
        for ei, ch in enumerate(EMNIST_IDX2CHAR)
        if ch in CHAR2IDX
    }
    _val_by_clf: dict[int, list[int]] = {}
    for i, lbl in enumerate(_emnist_val_ds.targets):
        ci = _emnist_to_clf.get(int(lbl))
        if ci is not None:
            _val_by_clf.setdefault(ci, []).append(i)
 
    for ci, indices in _val_by_clf.items():
        chosen = random.sample(indices, min(len(indices), EMNIST_VAL_PER_CLASS))
        for idx in chosen:
            img_pil, _ = _emnist_val_ds[idx]
            arr = np.array(img_pil)
            arr = cv2.flip(cv2.transpose(arr), flipCode=1)
            arr = cv2.bitwise_not(arr)   # fondo claro, tinta oscura
            rgb = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
            rsz = cv2.resize(rgb, (CFG['img_size'], CFG['img_size']), interpolation=cv2.INTER_CUBIC)
            val_emnist_samples.append((rsz, ci))
 
    print(f"  Val EMNIST (test split): {len(val_emnist_samples):,} muestras, {len(_val_by_clf)} clases")
    print(f"  ✅ Val set limpio — solo 62 clases EMNIST, distribución consistente")
    USE_CLEAN_VAL = True
 
except Exception as e:
    print(f"  [WARN] EMNIST val no disponible: {e}")
    print(f"  → Usando val_samples del split anterior")
    USE_CLEAN_VAL = False
    val_emnist_samples = val_samples   # fallback

## Celda 5 — Augmentation pipeline

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
IMG_SIZE = CFG['img_size']
 
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.OneOf([
        A.Affine(scale=(0.85,1.15),
                 translate_percent={"x":(-0.08,0.08),"y":(-0.08,0.08)},
                 rotate=(-15,15), shear=(-8,8), p=1.0),
        A.Perspective(scale=(0.02,0.06), p=1.0),
        A.ElasticTransform(alpha=1, sigma=10, p=1.0),
    ], p=0.80),
    A.OneOf([
        A.GaussNoise(std_range=(0.01,0.06), p=1.0),
        A.GaussianBlur(blur_limit=(3,5), p=1.0),
        A.MotionBlur(blur_limit=3, p=1.0),
    ], p=0.45),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.55),
    A.RandomGamma(gamma_limit=(80,120), p=0.30),
    A.CoarseDropout(num_holes_range=(1,3), hole_height_range=(4,12),
                    hole_width_range=(4,12), fill=255, p=0.20),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])
 
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

## Celda 6 — Dataset class y DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedShuffleSplit
 
class CropDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img, label = self.samples[idx]
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label
 
# Split de test desde train_samples (10%)
_labels_arr = np.array([s[1] for s in train_samples])
_min_count   = min(Counter(_labels_arr).values())
 
if _min_count >= 2:
    _sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
    _tr_idx, _te_idx = next(_sss.split(train_samples, _labels_arr))
else:
    _idx = list(range(len(train_samples))); random.shuffle(_idx)
    _cut = int(len(_idx) * 0.10)
    _te_idx, _tr_idx = _idx[:_cut], _idx[_cut:]
 
tr_smp = [train_samples[i] for i in _tr_idx]
te_smp = [train_samples[i] for i in _te_idx]
 
train_ds = CropDataset(tr_smp,               train_transform)
val_ds   = CropDataset(val_emnist_samples,   val_transform)   # ← SOLO EMNIST
test_ds  = CropDataset(te_smp,               val_transform)
 
print(f"\nSplits → Train:{len(train_ds):,}  Val:{len(val_ds):,} (EMNIST)  Test:{len(test_ds):,}")
 
# WeightedRandomSampler
tr_labels   = [s[1] for s in tr_smp]
cls_counts  = Counter(tr_labels)
w_per_class = 1.0 / np.array([max(cls_counts.get(i,1),1) for i in range(NUM_CLASSES)])
sample_w    = np.array([w_per_class[l] for l in tr_labels])
sampler     = WeightedRandomSampler(torch.FloatTensor(sample_w), len(train_ds)*2, replacement=True)
 
_lkw = dict(num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'])
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          sampler=sampler, drop_last=True, **_lkw)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size']*2, shuffle=False, **_lkw)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, **_lkw)
print(f"Batches → train:{len(train_loader):,}  val:{len(val_loader):,}")

## Celda 7 — Modelo EfficientNet-B2

In [ ]:
import timm
 
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
 
class SpanishOCRModel(nn.Module):
    def __init__(self, num_classes, pretrained=True, drop_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'], pretrained=pretrained,
            num_classes=0, global_pool='avg', drop_rate=drop_rate)
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim), nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512), nn.GELU(),
            nn.Dropout(drop_rate/2), nn.Linear(512, num_classes),
        )
    def forward(self, x): return self.head(self.backbone(x))
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
 
model = SpanishOCRModel(NUM_CLASSES, pretrained=CFG['pretrained'], drop_rate=CFG['drop_rate'])
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)
 
def _unwrap(m):
    while hasattr(m, 'module') and isinstance(m.module, nn.Module): m = m.module
    return m
_net = _unwrap(model).to(DEVICE)
_net.freeze_backbone()
 
print(f"Modelo: {sum(p.numel() for p in model.parameters()):,} params | {NUM_CLASSES} clases")

In [ ]:
# Sobreescribir TRAIN_CFG si ya existe
TRAIN_CFG = dict(
    lr_head      = 1e-3,
    lr_backbone  = 5e-5,
    weight_decay = 2e-4,          # más weight decay → menos overfitting
    label_smoothing = 0.10,       # más suavizado → menos overfitting (era 0.05)
    drop_rate       = 0.40,       # más dropout (se aplica al modelo abajo)
    epochs_warmup   = 10,
    epochs_total    = 120,
    patience        = 25,
    grad_clip       = 0.5,
    use_amp         = True,
    mixup_alpha     = 0.3,
    cutmix_alpha    = 0.7,
)
 
print(f"\n  TRAIN_CFG actualizado:")
print(f"  label_smoothing : {TRAIN_CFG['label_smoothing']} (más suavizado)")
print(f"  drop_rate       : {TRAIN_CFG['drop_rate']} (más dropout)")
print(f"  weight_decay    : {TRAIN_CFG['weight_decay']}")
print(f"  mixup/cutmix    : {TRAIN_CFG['mixup_alpha']}/{TRAIN_CFG['cutmix_alpha']}")

## Celda 8 — Optimizer, Scheduler, Loss

In [ ]:
def _make_plateau_scheduler(opt):
    """
    ReduceLROnPlateau: baja el LR cuando val_acc deja de mejorar.
    Más robusto que Cosine para fine-tuning en dos fases.
    """
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode     = 'max',   # maximizar val_acc
        factor   = 0.5,     # multiplicar LR por 0.5 al plateau
        patience = 8,       # esperar 8 épocas antes de bajar LR
        min_lr   = 1e-7,
        verbose  = True,
    )
 
# Monkey-patch _make_cosine_scheduler → _make_plateau_scheduler
# (para que la celda de entrenamiento lo use automáticamente)
_make_cosine_scheduler = lambda opt, epochs_remaining: _make_plateau_scheduler(opt)
print("\n  Scheduler: CosineAnnealingLR → ReduceLROnPlateau")
 
# Nota: en el loop de entrenamiento la llamada a scheduler.step() debe cambiar a:
#   scheduler.step(va_acc)   ← pasarle la métrica a monitorear
# Esto se indica en la celda de entrenamiento con el parche de abajo.
print("\n  ⚠  En el loop de entrenamiento cambia:")
print("     scheduler.step()  →  scheduler.step(va_acc)")
print("     (pasa el val_acc como argumento para ReduceLROnPlateau)")

## Celda 9 — MixUp / CutMix

In [ ]:
def mixup_batch(x, y, alpha=TRAIN_CFG['mixup_alpha']):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam
 
def cutmix_batch(x, y, alpha=TRAIN_CFG['cutmix_alpha']):
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    B,C,H,W = x.shape
    ch=int(H*np.sqrt(1-lam)); cw=int(W*np.sqrt(1-lam))
    cx=np.random.randint(W); cy=np.random.randint(H)
    x1=np.clip(cx-cw//2,0,W); x2=np.clip(cx+cw//2,0,W)
    y1=np.clip(cy-ch//2,0,H); y2=np.clip(cy+ch//2,0,H)
    x=x.clone(); x[:,:,y1:y2,x1:x2]=x[idx,:,y1:y2,x1:x2]
    return x, y, y[idx], 1-((x2-x1)*(y2-y1)/(W*H))
 
def mix_loss(p,ya,yb,lam): return lam*criterion(p,ya)+(1-lam)*criterion(p,yb)

## Celda 10 — Entrenamiento Clasificador + MLflow

In [ ]:
from torch.amp import autocast
from tqdm.notebook import tqdm
import mlflow
 
def train_epoch(net, loader, opt, scaler, epoch):
    net.train()
    total_loss = correct = total = 0
    for imgs, labels in tqdm(loader, desc=f'Train E{epoch+1}', leave=False):
        imgs   = imgs.to(DEVICE,   non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        r = np.random.rand()
        if r < 0.20:   imgs, ya, yb, lam = mixup_batch(imgs, labels);  use_mix=True
        elif r < 0.40: imgs, ya, yb, lam = cutmix_batch(imgs, labels); use_mix=True
        else: use_mix=False
        opt.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=TRAIN_CFG['use_amp']):
            out  = net(imgs)
            loss = mix_loss(out,ya,yb,lam) if use_mix else criterion(out,labels)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(net.parameters(), TRAIN_CFG['grad_clip'])
        scaler.step(opt); scaler.update()
        total_loss += loss.item()*imgs.size(0)
        if not use_mix:
            correct += (out.argmax(1)==labels).sum().item()
            total   += labels.size(0)
    return total_loss/len(loader.dataset), correct/max(total,1)
 
@torch.no_grad()
def eval_epoch(net, loader, desc='Val'):
    net.eval()
    tl=correct=top5=total=0
    for imgs, labels in tqdm(loader, desc=desc, leave=False):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=TRAIN_CFG['use_amp']):
            out = net(imgs); loss = criterion(out,labels)
        tl      += loss.item()*imgs.size(0)
        correct += (out.argmax(1)==labels).sum().item()
        k        = min(5,NUM_CLASSES)
        top5    += (out.topk(k,1)[1]==labels.unsqueeze(1)).any(1).sum().item()
        total   += labels.size(0)
    return tl/len(loader.dataset), correct/total, top5/total
 
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "═"*65)
print("  ENTRENAMIENTO CLASIFICADOR v3")
print("═"*65)
print(f"  Val set       : EMNIST test split ({len(val_ds):,} muestras, 62 clases)")
print(f"  Warm-up       : {TRAIN_CFG['epochs_warmup']} épocas (backbone frozen)")
print(f"  Fine-tuning   : {TRAIN_CFG['epochs_total']-TRAIN_CFG['epochs_warmup']} épocas (todo el modelo)")
print(f"  Total         : {TRAIN_CFG['epochs_total']} épocas")
print(f"  Patience      : {TRAIN_CFG['patience']} épocas\n")
 
mlflow.set_tracking_uri(CFG['mlflow_uri'])
mlflow.set_experiment(CFG['mlflow_experiment'])
 
best_val_acc = 0.0
best_ckpt    = str(CKPT_DIR / 'best_classifier_v3.pt')
best_state   = None
patience_cnt = 0
 
# history ahora usa 'lr' (compatible con celda de gráficas)
history = {k:[] for k in ('train_loss','val_loss','train_acc','val_acc','val_top5','lr')}
 
with mlflow.start_run(run_name='classifier_v3') as clf_run:
    mlflow.log_params({
        'model':CFG['model_name'], 'epochs':TRAIN_CFG['epochs_total'],
        'epochs_warmup':TRAIN_CFG['epochs_warmup'],
        'lr_head':TRAIN_CFG['lr_head'], 'lr_backbone':TRAIN_CFG['lr_backbone'],
        'label_smoothing':TRAIN_CFG['label_smoothing'],
        'val_source':'EMNIST_test_split',
        'n_train':len(train_ds), 'n_val':len(val_ds),
        'num_classes':NUM_CLASSES,
    })
 
    for epoch in range(TRAIN_CFG['epochs_total']):
        t0 = time.time()
 
        # ── Transición warm-up → fine-tuning ─────────────────────────────────
        if epoch == TRAIN_CFG['epochs_warmup']:
            print(f"\n  📢 Época {epoch+1}: DESCONGELANDO backbone → ReduceLROnPlateau\n")
            _net.unfreeze_backbone()
            optimizer        = _build_optimizer_finetune(_net)
            scheduler        = _make_plateau_scheduler(optimizer)
            scaler           = GradScaler('cuda', enabled=TRAIN_CFG['use_amp'])
            _in_finetune_phase = True
 
        tr_loss, tr_acc        = train_epoch(_net, train_loader, optimizer, scaler, epoch)
        va_loss, va_acc, va_t5 = eval_epoch(_net, val_loader)
 
        # ReduceLROnPlateau necesita la métrica; CosineAnnealingLR no
        if _in_finetune_phase:
            scheduler.step(va_acc)
        else:
            scheduler.step()
 
        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time()-t0
 
        for k,v in [('train_loss',tr_loss),('val_loss',va_loss),
                    ('train_acc',tr_acc),('val_acc',va_acc),
                    ('val_top5',va_t5),('lr',lr_now)]:
            history[k].append(v)
 
        mlflow.log_metrics({
            'train/loss':tr_loss,'train/acc':tr_acc,
            'val/loss':va_loss,'val/acc_top1':va_acc,'val/acc_top5':va_t5,
            'lr':lr_now,'epoch_sec':elapsed,
        }, step=epoch)
 
        phase = "WARM" if epoch < TRAIN_CFG['epochs_warmup'] else "FINE"
        print(f"  [{phase}] E{epoch+1:3d}/{TRAIN_CFG['epochs_total']} | "
              f"tr={tr_loss:.4f}/{tr_acc:.3f} | "
              f"va={va_loss:.4f}/{va_acc:.4f} top5={va_t5:.4f} | "
              f"lr={lr_now:.2e} | {elapsed:.1f}s")
 
        # Guardar mejor estado (solo si va_acc mejoró)
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            patience_cnt = 0
            best_state   = {k: v.cpu().clone() for k,v in _net.state_dict().items()}
            print(f"    ✅ Nuevo best val_acc = {best_val_acc:.4f}  (guardado en memoria)")
        else:
            patience_cnt += 1
            if patience_cnt >= TRAIN_CFG['patience']:
                print(f"\n  ⏹  Early stopping — {TRAIN_CFG['patience']} épocas sin mejora en val")
                break
 
    # Restaurar y guardar el MEJOR estado (no el último)
    if best_state is not None:
        _net.load_state_dict(best_state)
        print(f"\n  ← Modelo restaurado al mejor estado (val_acc={best_val_acc:.4f})")
 
    torch.save({
        'epoch':epoch, 'model_state':_net.state_dict(),
        'best_val_acc':best_val_acc, 'num_classes':NUM_CLASSES,
    }, best_ckpt)
    print(f"  💾 Checkpoint: {best_ckpt}")
 
    mlflow.log_metrics({'final/best_val_acc':best_val_acc})
    mlflow.log_artifact(best_ckpt, artifact_path='classifier')
    mlflow.log_artifact(str(char_map_path), artifact_path='classifier')
    CLF_RUN_ID = clf_run.info.run_id
 
print(f"\n✅ Entrenamiento completado | best_val_acc={best_val_acc:.4f}")
 
# ── Evaluación final en test ─────────────────────────────────────────────────
print("\n=== Evaluación final en Test set ===")
te_loss, te_acc, te_top5 = eval_epoch(_net, test_loader, desc='Test')
print(f"  Top-1 : {te_acc:.4f}  ({te_acc*100:.2f}%)")
print(f"  Top-5 : {te_top5:.4f}  ({te_top5*100:.2f}%)")

## Celda 11 — Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history['train_loss']) + 1)
 
axes[0].plot(ep, history['train_loss'], label='Train')
axes[0].plot(ep, history['val_loss'],   label='Val (EMNIST)')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
 
axes[1].plot(ep, history['train_acc'], label='Train')
axes[1].plot(ep, history['val_acc'],   label='Val Top-1 (EMNIST)')
axes[1].plot(ep, history['val_top5'],  label='Val Top-5', linestyle='--')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
 
# 'lr' en lugar de 'lr_head' — compatible con la clave que usa esta celda
axes[2].plot(ep, history['lr'])
axes[2].set_title('Learning Rate'); axes[2].set_yscale('log'); axes[2].grid(True)
 
# Línea de separación warm-up / fine-tuning
wu = TRAIN_CFG['epochs_warmup']
for ax in axes:
    if len(ep) > wu:
        ax.axvline(wu, color='orange', ls='--', lw=1, alpha=0.7, label='Unfreeze')
 
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves_v3.png'), dpi=150)
plt.show()
print(f"Curvas guardadas → {OUTPUT_DIR}/training_curves_v3.png")

## Celda 12 — Evaluacion final en test

## Celda 13 — Exportar Clasificador a ONNX + INT8

In [ ]:
import onnx, onnxruntime as ort
from onnxsim import simplify

onnx_fp32 = str(ONNX_DIR / 'classifier_fp32.onnx')
onnx_sim  = str(ONNX_DIR / 'classifier_simplified.onnx')
onnx_int8 = str(ONNX_DIR / 'classifier_int8.onnx')

eval_model.eval().cpu()
dummy = torch.randn(1, 3, CFG['img_size'], CFG['img_size'])

print('Exportando ONNX FP32 ...')
torch.onnx.export(
    eval_model, dummy, onnx_fp32,
    export_params=True, opset_version=17, do_constant_folding=True, dynamo=False,
    input_names=['image'], output_names=['logits'],
    dynamic_axes={'image':{0:'batch'},'logits':{0:'batch'}})
print(f'  FP32 -> {os.path.getsize(onnx_fp32)/1e6:.1f} MB')

try:
    m_sim, ok = simplify(onnx.load(onnx_fp32))
    if ok: onnx.save(m_sim, onnx_sim); print(f'  Simplified -> {os.path.getsize(onnx_sim)/1e6:.1f} MB')
    else:  shutil.copy(onnx_fp32, onnx_sim)
except Exception as e:
    print(f'  Simplify fallo ({e}); copiando FP32'); shutil.copy(onnx_fp32, onnx_sim)

try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(onnx_sim, onnx_int8, weight_type=QuantType.QInt8)
    print(f'  INT8 -> {os.path.getsize(onnx_int8)/1e6:.1f} MB')
except Exception as e:
    print(f'  INT8 fallo: {e}')

with torch.no_grad():
    pt_out = eval_model(dummy).numpy()
sess = ort.InferenceSession(onnx_sim, providers=['CPUExecutionProvider'])
ort_out = sess.run(None, {'image': dummy.numpy()})[0]
print(f'  Max diff PyTorch vs ONNX: {np.abs(pt_out-ort_out).max():.6f}')

with mlflow.start_run(run_id=CLF_RUN_ID):
    for p in [onnx_fp32, onnx_sim, onnx_int8]:
        if os.path.exists(p): mlflow.log_artifact(p, artifact_path='onnx')
    mlflow.log_metrics({'test/top1':te_acc,'test/top5':te_top5})
print('Artefactos ONNX clasificador en MLflow')


## Celda 14 — Entrenar Detector YOLOv8n + MLflow

## Celda 15 — Exportar Detector a ONNX

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# --- RUTA DIRECTA PROPORCIONADA ---
yolo_best_pt = "/kaggle/working/runs/detect/char_detector_t4_fast/weights/best.pt"

yolo_onnx_path = None

if Path(yolo_best_pt).exists():
    print(f'✅ Archivo encontrado, iniciando exportación: {yolo_best_pt}')
    
    # Cargar el modelo desde la ruta directa
    det_ex = YOLO(yolo_best_pt)
    
    # Exportar a ONNX
    # Usamos imgsz de YOLO_CFG si existe, de lo contrario 640 por defecto
    img_size = YOLO_CFG.get('img_size', 640)
    
    print(f'  Exportando a ONNX (imgsz={img_size})...')
    export_path = det_ex.export(
        format='onnx', 
        imgsz=img_size,
        opset=17, 
        dynamic=False, 
        simplify=True
    )
    
    # El método export devuelve la ruta del archivo generado
    yolo_onnx_path = str(export_path)
    
    if Path(yolo_onnx_path).exists():
        size_mb = os.path.getsize(yolo_onnx_path) / 1e6
        print(f'  🚀 ONNX generado con éxito: {yolo_onnx_path} ({size_mb:.2f} MB)')
    else:
        print('  ❌ Error: El proceso de exportación no generó el archivo esperado.')
        yolo_onnx_path = None
else:
    print(f'❌ ERROR: No se encontró el archivo en la ruta: {yolo_best_pt}')
    print('Verifica si el nombre de la carpeta "char_detector_t4_fast" es correcto.')

# --- LOG EN MLFLOW ---
# Solo si existe el run_id y el archivo ONNX
if 'yolo_cb' in globals() and yolo_cb.run_id and yolo_onnx_path and Path(yolo_onnx_path).exists():
    import mlflow
    with mlflow.start_run(run_id=yolo_cb.run_id):
        print(f'📦 Registrando artefactos en MLflow (Run: {yolo_cb.run_id})...')
        mlflow.log_artifact(yolo_onnx_path, artifact_path='onnx')
        mlflow.log_artifact(yolo_best_pt, artifact_path='weights')
    print('✅ Artefactos YOLO registrados en MLflow.')

## Celda 16 — Copiar artefactos finales

In [ ]:
FINAL_DIR = OUTPUT_DIR / 'final_models'
FINAL_DIR.mkdir(exist_ok=True)

def _cp(src, name):
    src = Path(src) if src else None
    if src and src.exists():
        dst = FINAL_DIR/name; shutil.copy2(src, dst)
        print(f'  {name}  ({dst.stat().st_size/1e6:.1f} MB)'); return str(dst)
    print(f'  !! {name}: no encontrado'); return None

print('=== Copiando modelos finales ===')
_cp(onnx_sim,          'mobilenet_classifier.onnx')
_cp(onnx_int8,         'mobilenet_classifier_int8.onnx')
_cp(yolo_onnx_path,    'yolo_detector.onnx')
_cp(char_map_path,     'char_map.json')
_cp(best_ckpt,         'best_classifier.pt')
if yolo_best_pt: _cp(yolo_best_pt, 'best_detector.pt')

print(f'\nContenido de {FINAL_DIR}:')
for f in sorted(FINAL_DIR.iterdir()):
    print(f'  {f.name:45s}  {f.stat().st_size/1e6:.2f} MB')


## Celda 17 — Paquete ZIP + Verificacion Final

In [ ]:
import zipfile

metadata = {
    'project':'Tutor Inteligente OCR Espanol','version':'2.0.0','date':'2026-03',
    'classifier':{'model':CFG['model_name'],'num_classes':NUM_CLASSES,
                  'img_size':CFG['img_size'],'best_val_acc':round(best_val_acc,4),
                  'input':'image','output':'logits',
                  'preprocessing':{'resize':[CFG['img_size'],CFG['img_size']],
                                   'normalize':{'mean':[0.485,0.456,0.406],'std':[0.229,0.224,0.225]}}},
    'detector':{'model':YOLO_CFG['model_variant'],'img_size':YOLO_CFG['img_size'],'nc':1,'class':'trazo'},
    'mlflow_uri':CFG['mlflow_uri'],'onnx_opset':17,
    'recommended_clf':'mobilenet_classifier_int8.onnx',
    'recommended_det':'yolo_detector.onnx',
}
with open(FINAL_DIR/'model_metadata.json','w',encoding='utf-8') as f:
    json.dump(metadata,f,ensure_ascii=False,indent=2)

zip_path = str(OUTPUT_DIR/'tutor_inteligente_models.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(FINAL_DIR.iterdir()): zf.write(fp,fp.name)
    curves = OUTPUT_DIR/'training_curves.png'
    if curves.exists(): zf.write(curves,'training_curves.png')

print(f'ZIP: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
with zipfile.ZipFile(zip_path) as zf:
    for i in zf.infolist():
        print(f'  {i.filename:50s}  {i.file_size/1e6:.2f} MB')

# ---- Verificacion clasificador ----
print('\n=== Verificacion ONNX Clasificador ===')
clf_onnx_f = str(FINAL_DIR/'mobilenet_classifier.onnx')
if Path(clf_onnx_f).exists():
    s = ort.InferenceSession(clf_onnx_f, providers=['CPUExecutionProvider'])
    ti = np.random.randn(1,3,CFG['img_size'],CFG['img_size']).astype(np.float32)
    out = s.run(None,{'image':ti})[0]
    pid = int(np.argmax(out[0]))
    print(f'  Shape out: {out.shape}  |  pred_idx={pid}  char={repr(IDX2CHAR.get(pid,"?"))}')
    print('  OK')

# ---- Verificacion detector ----
print('\n=== Verificacion ONNX Detector ===')
det_onnx_f = str(FINAL_DIR/'yolo_detector.onnx')
if Path(det_onnx_f).exists():
    sd = ort.InferenceSession(det_onnx_f, providers=['CPUExecutionProvider'])
    inp_n = sd.get_inputs()[0].name
    td = np.random.randn(1,3,640,640).astype(np.float32)
    od = sd.run(None,{inp_n:td})[0]
    print(f'  Input: {td.shape}  Output: {od.shape}')
    print('  OK')
else:
    print('  yolo_detector.onnx no encontrado')

print('\n' + '='*55)
print('PIPELINE COMPLETADO')
print(f'  Clasificador best acc : {best_val_acc*100:.2f}%')
print(f'  MLflow tracking      : {CFG["mlflow_uri"]}')
print(f'  ZIP final            : {zip_path}')
print('='*55)


In [ ]:
from IPython.display import FileLink

# Genera el enlace de descarga
FileLink(r'tutor_inteligente_models.zip')